# 3D Sensitivity Analysis — GBM Hyperparameters

This notebook sweeps key GBM-3D hyperparameters (overlap, lr, epochs) on a synthetic 3D volume and reports mean sensitivity loss.

It mirrors the 2D sensitivity analysis from the paper but for the 3D (D-1) case.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from gbm.core import find_optimal_k_points_gbm_3D

print("3D sensitivity sweep notebook")

In [ ]:
# Small synthetic 3D volume
lw = 2
W, H_img, D = 100 * lw, 100, 8
x = np.linspace(-20, 20, W)
y = np.linspace(0.4, 12, D)
z = np.linspace(-70, 70, H_img)
X, Y, Z = np.meshgrid(x, y, z, indexing="ij")
C = 5e-6 + 3e-6 * np.exp(-((X)**2 + (Z)**2 + (Y-5)**2) / 55)

df = pd.DataFrame({"X": X.ravel(), "Y": Y.ravel(), "Z": Z.ravel(), "Carbon": C.ravel()})
inside = np.ones(W * H_img * D, dtype=bool)
gmean = float(C.mean())
print(f"Volume ready, mean = {gmean:.2e}")

In [ ]:
k = 5
results = {"overlap": {}, "lr": {}, "epochs": {}}

# Sweep overlap
for ov in [30, 50, 75, 90]:
    loss, m, s, _ = find_optimal_k_points_gbm_3D(
        df, inside, k=k, in_CO2_avg=gmean,
        cross_section="XZ", barn_LW_ratio=lw,
        overlap=ov, lr=5e-7, epochs=6, sampling_budget=400
    )
    results["overlap"][str(ov)] = {"mean": m, "std": s}
    print(f"overlap={ov}: mean_sens={m:.2e}")

In [ ]:
# Sweep lr (log scale)
for lr in [1e-8, 5e-7, 1e-6, 1e-5]:
    loss, m, s, _ = find_optimal_k_points_gbm_3D(
        df, inside, k=k, in_CO2_avg=gmean,
        cross_section="XZ", barn_LW_ratio=lw,
        overlap=75, lr=lr, epochs=6, sampling_budget=400
    )
    results["lr"][str(lr)] = {"mean": m, "std": s}
    print(f"lr={lr:.0e}: mean_sens={m:.2e}")

In [ ]:
# Sweep epochs
for ep in [3, 6, 10, 15]:
    loss, m, s, _ = find_optimal_k_points_gbm_3D(
        df, inside, k=k, in_CO2_avg=gmean,
        cross_section="XZ", barn_LW_ratio=lw,
        overlap=75, lr=5e-7, epochs=ep, sampling_budget=400
    )
    results["epochs"][str(ep)] = {"mean": m, "std": s}
    print(f"epochs={ep}: mean_sens={m:.2e}")

In [ ]:
# Simple plot
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
for ax, param in zip(axes, ["overlap", "lr", "epochs"]):
    vals = results[param]
    xs = [float(k) for k in vals.keys()]
    ys = [v["mean"] for v in vals.values()]
    ax.plot(xs, ys, "o-")
    ax.set_xlabel(param)
    ax.set_ylabel("Mean sensitivity loss")
    if param == "lr":
        ax.set_xscale("log")
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("3d_sensitivity_sweep.png", dpi=150)
print("Saved 3d_sensitivity_sweep.png")